In [0]:
%py
from pyspark.sql import Row

orders = [
    ("1001", "C001", "P001", 2, "2026-08-01", "COMPLETED"),
    ("1002", "C002", "P003", 1, "2026-08-01", "COMPLETED"),
    ("1003", "C001", "P002", 4, "2026-08-02", "CANCELLED"),
    ("1004", "C003", "P001", 1, "2026-08-02", "COMPLETED"),
    ("1005", "C002", "P002", 2, "2026-08-03", "COMPLETED"),
    ("1006", "C004", "P003", 3, "2026-08-03", "COMPLETED"),
    ("1007", "C001", "P001", 1, "2026-08-04", "COMPLETED")
]

columns = [
    "order_id",
    "customer_id",
    "product_id",
    "quantity",
    "order_date",
    "status"
]

orders_df = spark.createDataFrame(orders, columns)

display(orders_df)

order_id,customer_id,product_id,quantity,order_date,status
1001,C001,P001,2,2026-08-01,COMPLETED
1002,C002,P003,1,2026-08-01,COMPLETED
1003,C001,P002,4,2026-08-02,CANCELLED
1004,C003,P001,1,2026-08-02,COMPLETED
1005,C002,P002,2,2026-08-03,COMPLETED
1006,C004,P003,3,2026-08-03,COMPLETED
1007,C001,P001,1,2026-08-04,COMPLETED


In [0]:
customers = [
    ("C001", "Shivam", "shivam@example.com", "Pune", "2026-01-10"),
    ("C002", "Rahul", "rahul@example.com", "Mumbai", "2026-02-15"),
    ("C003", "Amit", "amit@example.com", "Delhi", "2026-03-20"),
    ("C004", "Neha", "neha@example.com", "Bangalore", "2026-04-05")
]

customer_columns = [
    "customer_id",
    "customer_name",
    "email",
    "city",
    "signup_date"
]

customers_df = spark.createDataFrame(
    customers,
    customer_columns
)

display(customers_df)

customer_id,customer_name,email,city,signup_date
C001,Shivam,shivam@example.com,Pune,2026-01-10
C002,Rahul,rahul@example.com,Mumbai,2026-02-15
C003,Amit,amit@example.com,Delhi,2026-03-20
C004,Neha,neha@example.com,Bangalore,2026-04-05


In [0]:
products = [
    ("P001", "Laptop", "Electronics", 60000.0),
    ("P002", "Headphones", "Electronics", 3000.0),
    ("P003", "Keyboard", "Accessories", 1500.0)
]

product_columns = [
    "product_id",
    "product_name",
    "category",
    "price"
]

products_df = spark.createDataFrame(
    products,
    product_columns
)

display(products_df)

product_id,product_name,category,price
P001,Laptop,Electronics,60000.0
P002,Headphones,Electronics,3000.0
P003,Keyboard,Accessories,1500.0


In [0]:
orders_df.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("ecommerce.production.bronze_orders")

In [0]:
customers_df.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("ecommerce.production.bronze_customers")

In [0]:
products_df.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("ecommerce.production.bronze_products")

In [0]:
%sql
SELECT *
FROM ecommerce.production.bronze_orders;

order_id,customer_id,product_id,quantity,order_date,status
1001,C001,P001,2,2026-08-01,COMPLETED
1002,C002,P003,1,2026-08-01,COMPLETED
1003,C001,P002,4,2026-08-02,CANCELLED
1004,C003,P001,1,2026-08-02,COMPLETED
1005,C002,P002,2,2026-08-03,COMPLETED
1006,C004,P003,3,2026-08-03,COMPLETED
1007,C001,P001,1,2026-08-04,COMPLETED


In [0]:
orders = spark.table(
    "ecommerce.production.bronze_orders"
)

customers = spark.table(
    "ecommerce.production.bronze_customers"
)

products = spark.table(
    "ecommerce.production.bronze_products"
)

In [0]:
from pyspark.sql.functions import col, to_date

silver_orders = (
    orders
    .withColumn("quantity", col("quantity").cast("integer"))
    .withColumn("order_date", to_date("order_date"))
    .filter(col("order_id").isNotNull())
    .filter(col("customer_id").isNotNull())
    .filter(col("product_id").isNotNull())
)

In [0]:
silver_orders = silver_orders.dropDuplicates(
    ["order_id"]
)

In [0]:
silver_orders = silver_orders.filter(
    col("status").isin(
        "COMPLETED",
        "CANCELLED"
    )
)

In [0]:
silver_orders.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable(
        "ecommerce.production.silver_orders"
    )

In [0]:
from pyspark.sql.functions import col

sales = (
    silver_orders
    .join(
        products,
        "product_id",
        "left"
    )
    .join(
        customers,
        "customer_id",
        "left"
    )
)

In [0]:
sales = sales.withColumn(
    "revenue",
    col("quantity") * col("price")
)

In [0]:
display(sales)

customer_id,product_id,order_id,quantity,order_date,status,product_name,category,price,customer_name,email,city,signup_date,revenue
C001,P002,1003,4,2026-08-02,CANCELLED,Headphones,Electronics,3000.0,Shivam,shivam@example.com,Pune,2026-01-10,12000.0
C002,P002,1005,2,2026-08-03,COMPLETED,Headphones,Electronics,3000.0,Rahul,rahul@example.com,Mumbai,2026-02-15,6000.0
C001,P001,1007,1,2026-08-04,COMPLETED,Laptop,Electronics,60000.0,Shivam,shivam@example.com,Pune,2026-01-10,60000.0
C001,P001,1001,2,2026-08-01,COMPLETED,Laptop,Electronics,60000.0,Shivam,shivam@example.com,Pune,2026-01-10,120000.0
C003,P001,1004,1,2026-08-02,COMPLETED,Laptop,Electronics,60000.0,Amit,amit@example.com,Delhi,2026-03-20,60000.0
C002,P003,1002,1,2026-08-01,COMPLETED,Keyboard,Accessories,1500.0,Rahul,rahul@example.com,Mumbai,2026-02-15,1500.0
C004,P003,1006,3,2026-08-03,COMPLETED,Keyboard,Accessories,1500.0,Neha,neha@example.com,Bangalore,2026-04-05,4500.0


In [0]:
from pyspark.sql.functions import sum, count

daily_revenue = (
    sales
    .filter(col("status") == "COMPLETED")
    .groupBy("order_date")
    .agg(
        sum("revenue").alias("total_revenue"),
        count("order_id").alias("total_orders")
    )
)

In [0]:
daily_revenue.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable(
        "ecommerce.production.gold_daily_revenue"
    )

In [0]:
%sql
SELECT *
FROM ecommerce.production.gold_daily_revenue
ORDER BY order_date;

order_date,total_revenue,total_orders
2026-08-01,121500.0,2
2026-08-02,60000.0,1
2026-08-03,10500.0,2
2026-08-04,60000.0,1


In [0]:
product_sales = (
    sales
    .filter(col("status") == "COMPLETED")
    .groupBy(
        "product_id",
        "product_name",
        "category"
    )
    .agg(
        sum("quantity").alias("units_sold"),
        sum("revenue").alias("total_revenue")
    )
)

In [0]:
product_sales.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable(
        "ecommerce.production.gold_product_sales"
    )

In [0]:
customer_sales = (
    sales
    .filter(col("status") == "COMPLETED")
    .groupBy(
        "customer_id",
        "customer_name",
        "city"
    )
    .agg(
        count("order_id").alias("total_orders"),
        sum("revenue").alias("total_spent")
    )
)

In [0]:
customer_sales.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable(
        "ecommerce.production.gold_customer_sales"
    )

In [0]:
%sql
SHOW TABLES IN ecommerce.production;

database,tableName,isTemporary
production,bronze_customers,false
production,bronze_orders,false
production,bronze_products,false
production,gold_customer_sales,false
production,gold_daily_revenue,false
production,gold_product_sales,false
production,silver_orders,false


In [0]:
%sql
SHOW TABLES IN workspace.default;

database,tableName,isTemporary
default,gold_daily_revenue_pipeline,false
default,silver_orders_pipeline,false


In [0]:
%sql
SELECT *
FROM workspace.default.silver_orders_pipeline;

order_id,customer_id,product_id,quantity,order_date,status
1007,C001,P001,1,2026-08-04,COMPLETED
1006,C004,P003,3,2026-08-03,COMPLETED
1002,C002,P003,1,2026-08-01,COMPLETED
1004,C003,P001,1,2026-08-02,COMPLETED
1001,C001,P001,2,2026-08-01,COMPLETED
1005,C002,P002,2,2026-08-03,COMPLETED
1003,C001,P002,4,2026-08-02,CANCELLED


In [0]:
%sql
SELECT *
FROM workspace.default.gold_daily_revenue_pipeline;

order_date,total_revenue,total_orders
2026-08-02,60000.0,1
2026-08-04,60000.0,1
2026-08-01,121500.0,2
2026-08-03,10500.0,2
